# Exp 014: HGNetV2 Pseudolabel

First pseudo-label continuation of `exp_011`. The goal is to keep the strong `HGNetV2` supervised branch intact, use the existing `4-fold exp_011` checkpoints as teachers on unlabeled `train_soundscapes`, and test whether target-domain pseudo labels can push the native branch beyond the stabilized `0.850` Kaggle baseline.


## Plan

1. Rebuild the `exp_011` labeled training dataframe (`train_audio + labeled soundscape clips`).
2. Load `exp_011` fold checkpoints as teachers, excluding the current student fold.
3. Generate soft pseudo labels on unlabeled soundscape files only.
4. Train a student `HGNetV2-B0` on labeled data plus pseudo-labeled windows.
5. Select checkpoints by `soundscape_macro_auc`, not by overall fold AUC.


In [11]:
from __future__ import annotations

import ast
import json
import math
import os
import random
import re
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
import typing as tp

import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import ConcatDataset, DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm


In [12]:
ROOT = Path('/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026')
DATA = ROOT / 'data' / 'birdclef-2026'
PROCESSED = ROOT / 'processed_data'
EXPERIMENTS = ROOT / 'experiments'
EXP011_DIR = EXPERIMENTS / 'outputs' / 'exp_011_hgnetv2_soundscape_supervised'


@dataclass
class Config:
    experiment_id: str = 'exp_014'
    experiment_name: str = 'hgnetv2_pseudolabel'
    fold: int = 3
    n_folds: int = 4
    random_seed: int = 1091

    sample_rate: int = 32_000
    segment_seconds: int = 5
    n_fft: int = 2048
    win_length: int = 626
    hop_length: int = 313
    f_min: int = 20
    n_mels: int = 256
    top_db: float = 80.0
    image_size: tuple[int, int] = (256, 256)

    model_name: str = 'hgnetv2_b0.ssld_stage2_ft_in1k'
    pretrained: bool = True
    drop_path_rate: float = 0.0

    epochs: int = 8
    batch_size: int = 16
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    num_workers: int = 0
    use_amp: bool = True

    pseudo_batch_size: int = 48
    pseudo_min_confidence: float = 0.30
    pseudo_power: float = 1.15
    pseudo_loss_weight: float = 0.50
    pseudo_sampler_power: float = 2.0
    max_pseudo_segments_per_file: int = 6
    max_pseudo_segments_total: int | None = None
    max_pseudo_files: int | None = None
    max_train_rows: int | None = None
    max_valid_rows: int | None = None
    use_mixup: bool = False
    mixup_alpha: float = 1.0
    mixup_theta: float = 0.8
    save_every_epoch: bool = True

    @property
    def clip_samples(self) -> int:
        return int(self.sample_rate * self.segment_seconds)


CFG = Config()
RUN_PREPARE = False
RUN_PSEUDO_GENERATION = False
RUN_TRAINING = True
RESUME_TRAINING = True

PROC_DIR = PROCESSED / CFG.experiment_id
OUTPUT_DIR = EXPERIMENTS / 'outputs' / f'{CFG.experiment_id}_{CFG.experiment_name}' / f'fold_{CFG.fold:02d}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)


In [13]:
def set_random_seed(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def autocast_context() -> tp.ContextManager[object]:
    if amp_enabled:
        return torch.amp.autocast('cuda', enabled=True)
    return nullcontext()


def save_json(payload: dict[str, tp.Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=False))


def sanitize_filename(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value)


set_random_seed(CFG.random_seed)
device = pick_device()
amp_enabled = bool(CFG.use_amp and device.type == 'cuda')
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

print({
    'root': str(ROOT),
    'device': str(device),
    'amp_enabled': amp_enabled,
    'output_dir': str(OUTPUT_DIR),
})


{'root': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026', 'device': 'mps', 'amp_enabled': False, 'output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_014_hgnetv2_pseudolabel/fold_03'}


In [14]:
TRAIN_AUDIO_DIR = DATA / 'train_audio'
TRAIN_SOUNDSCAPE_DIR = DATA / 'train_soundscapes'
TRAIN_AUDIO_WAV_DIR = PROC_DIR / 'train_audio_wav'
TRAIN_AUDIO_WAV_DIR.mkdir(parents=True, exist_ok=True)

train_labels = pd.read_csv(DATA / 'train.csv')
train_soundscape_segments = pd.read_csv(DATA / 'train_soundscapes_labels.csv').drop_duplicates().reset_index(drop=True)
taxonomy = pd.read_csv(DATA / 'taxonomy.csv')

CLASSES = taxonomy['primary_label'].tolist()
N_CLASSES = len(CLASSES)
label2idx = {label: idx for idx, label in enumerate(CLASSES)}

for col in ['start', 'end']:
    train_soundscape_segments[col] = pd.to_datetime(train_soundscape_segments[col], format='%H:%M:%S')
train_soundscape_segments['start_sec'] = train_soundscape_segments['start'].dt.minute * 60 + train_soundscape_segments['start'].dt.second
train_soundscape_segments['end_sec'] = train_soundscape_segments['end'].dt.minute * 60 + train_soundscape_segments['end'].dt.second

expanded_rows = []
for row in train_soundscape_segments.itertuples(index=False):
    labels = [token.strip() for token in str(row.primary_label).split(';') if token.strip() and token.strip() in label2idx]
    for label in labels:
        expanded_rows.append({
            'filename': row.filename,
            'start_sec': int(row.start_sec),
            'end_sec': int(row.end_sec),
            'primary_label': label,
        })
train_soundscape_labels = pd.DataFrame(expanded_rows)


def original_train_audio_path(filename: str) -> Path:
    return TRAIN_AUDIO_DIR / filename


def train_audio_cache_subdir(primary_label: str, filename: str) -> Path:
    rel = Path(filename)
    if len(rel.parts) > 1:
        return rel.parent
    return Path(primary_label)


def expected_train_audio_wav_path(primary_label: str, filename: str) -> Path:
    return TRAIN_AUDIO_WAV_DIR / train_audio_cache_subdir(primary_label, filename) / f"{Path(filename).stem}.wav"


def resolve_train_audio_path(primary_label: str, filename: str) -> str:
    wav_path = expected_train_audio_wav_path(primary_label, filename)
    if wav_path.exists():
        return str(wav_path)
    return str(original_train_audio_path(filename))


def merge_contiguous_soundscape_intervals(label_df: pd.DataFrame) -> pd.DataFrame:
    merged_rows: list[dict[str, tp.Any]] = []
    ordered = label_df.sort_values(['filename', 'primary_label', 'start_sec', 'end_sec']).reset_index(drop=True)
    for (filename, primary_label), group in ordered.groupby(['filename', 'primary_label'], sort=False):
        current_start = None
        current_end = None
        for row in group.itertuples(index=False):
            start_sec = int(row.start_sec)
            end_sec = int(row.end_sec)
            if current_start is None:
                current_start = start_sec
                current_end = end_sec
                continue
            if start_sec <= current_end:
                current_end = max(current_end, end_sec)
            else:
                merged_rows.append({
                    'audio_id': Path(filename).stem,
                    'filename': filename,
                    'primary_label': primary_label,
                    'labels': primary_label,
                    'source': 'soundscape_clip',
                    'source_file_path': str(TRAIN_SOUNDSCAPE_DIR / filename),
                    'start_sec': current_start,
                    'end_sec': current_end,
                })
                current_start = start_sec
                current_end = end_sec
        if current_start is not None:
            merged_rows.append({
                'audio_id': Path(filename).stem,
                'filename': filename,
                'primary_label': primary_label,
                'labels': primary_label,
                'source': 'soundscape_clip',
                'source_file_path': str(TRAIN_SOUNDSCAPE_DIR / filename),
                'start_sec': current_start,
                'end_sec': current_end,
            })
    manifest = pd.DataFrame(merged_rows)
    manifest['clip_start_frame'] = (manifest['start_sec'] * CFG.sample_rate).astype(int)
    manifest['clip_end_frame'] = (manifest['end_sec'] * CFG.sample_rate).astype(int)
    manifest['clip_duration_sec'] = manifest['end_sec'] - manifest['start_sec']
    manifest['file_path'] = manifest['source_file_path']
    manifest['cache_mode'] = 'offset_read'
    return manifest


def build_train_audio_frame(label_df: pd.DataFrame) -> pd.DataFrame:
    frame = pd.DataFrame({
        'audio_id': label_df['filename'].map(lambda x: Path(x).stem),
        'filename': label_df['filename'],
        'primary_label': label_df['primary_label'],
        'secondary_labels': label_df['secondary_labels'],
    })
    frame['labels'] = [
        ';'.join([primary] + [label for label in ast.literal_eval(secondary) if label in label2idx])
        for primary, secondary in frame[['primary_label', 'secondary_labels']].itertuples(index=False)
    ]
    frame['source'] = 'train_audio'
    frame['file_path'] = [resolve_train_audio_path(primary_label, filename) for primary_label, filename in frame[['primary_label', 'filename']].itertuples(index=False)]
    frame['source_file_path'] = [str(original_train_audio_path(filename)) for filename in frame['filename'].tolist()]
    frame['clip_start_frame'] = 0
    frame['clip_end_frame'] = -1
    frame['clip_duration_sec'] = np.nan
    frame['cache_mode'] = frame['file_path'].map(lambda x: 'wav_cache' if x.endswith('.wav') else 'ogg_offset')
    return frame[['audio_id', 'filename', 'primary_label', 'labels', 'source', 'file_path', 'source_file_path', 'clip_start_frame', 'clip_end_frame', 'clip_duration_sec', 'cache_mode']]


def build_multihot(labels: pd.Series, mapping: dict[str, int]) -> np.ndarray:
    arr = np.zeros((len(labels), len(mapping)), dtype=np.float32)
    for row_idx, label_string in enumerate(labels.tolist()):
        for label in str(label_string).split(';'):
            if label in mapping:
                arr[row_idx, mapping[label]] = 1.0
    return arr

soundscape_clip_manifest = merge_contiguous_soundscape_intervals(train_soundscape_labels)
train_audio_frame = build_train_audio_frame(train_labels)
labeled_df = pd.concat([
    train_audio_frame,
    soundscape_clip_manifest[['audio_id', 'filename', 'primary_label', 'labels', 'source', 'file_path', 'source_file_path', 'clip_start_frame', 'clip_end_frame', 'clip_duration_sec', 'cache_mode']],
], axis=0, ignore_index=True)
labeled_arr = build_multihot(labeled_df['labels'], label2idx)

print({
    'labeled_rows': int(len(labeled_df)),
    'train_audio_rows': int((labeled_df['source'] == 'train_audio').sum()),
    'soundscape_clip_rows': int((labeled_df['source'] == 'soundscape_clip').sum()),
    'n_classes': N_CLASSES,
})


{'labeled_rows': 36078, 'train_audio_rows': 35549, 'soundscape_clip_rows': 529, 'n_classes': 234}


In [15]:
class MultiLabelStratifiedGroupKFold:
    def __init__(self, n_splits: int, random_state: int):
        self.n_splits = n_splits
        self.random_state = random_state

    def split(self, label_arr: np.ndarray, group_ids: np.ndarray):
        np.random.seed(self.random_state)
        random.seed(self.random_state)

        unique_groups = sorted(set(group_ids))
        group_to_idx = {group_id: idx for idx, group_id in enumerate(unique_groups)}
        group_index_arr = np.vectorize(group_to_idx.get)(group_ids)

        n_groups = len(unique_groups)
        n_classes = label_arr.shape[1]
        class_totals = label_arr.sum(axis=0)
        class_totals[class_totals == 0] = 1

        counts_by_group = np.zeros((n_groups, n_classes), dtype=np.int64)
        for row_idx, group_idx in enumerate(group_index_arr):
            counts_by_group[group_idx] += label_arr[row_idx].astype(np.int64)

        counts_by_fold = np.zeros((self.n_splits, n_classes), dtype=np.int64)
        groups_by_fold: list[list[int]] = [[] for _ in range(self.n_splits)]

        group_items = list(enumerate(counts_by_group))
        random.shuffle(group_items)

        for group_idx, group_count in sorted(group_items, key=lambda item: -np.std(item[1])):
            best_fold = None
            best_score = None
            for fold_idx in range(self.n_splits):
                counts_by_fold[fold_idx] += group_count
                fold_score = (counts_by_fold / class_totals).std(axis=0).mean()
                counts_by_fold[fold_idx] -= group_count
                if best_score is None or fold_score < best_score:
                    best_score = fold_score
                    best_fold = fold_idx
            counts_by_fold[best_fold] += group_count
            groups_by_fold[best_fold].append(group_idx)

        row_indices = np.arange(label_arr.shape[0])
        for fold_idx in range(self.n_splits):
            valid_groups = groups_by_fold[fold_idx]
            valid_mask = np.isin(group_index_arr, valid_groups)
            yield row_indices[~valid_mask], row_indices[valid_mask]


splitter = MultiLabelStratifiedGroupKFold(n_splits=CFG.n_folds, random_state=CFG.random_seed)
splits = list(splitter.split(labeled_arr, labeled_df['audio_id'].to_numpy()))
labeled_df = labeled_df.copy()
labeled_df['fold'] = -1
for fold_idx, (_, valid_idx) in enumerate(splits):
    labeled_df.loc[valid_idx, 'fold'] = fold_idx


def build_fold_frame(frame: pd.DataFrame, label_arr: np.ndarray, fold_id: int):
    train_mask = frame['fold'] != fold_id
    valid_mask = frame['fold'] == fold_id

    train_frame = frame.loc[train_mask].reset_index(drop=True)
    valid_frame = frame.loc[valid_mask].reset_index(drop=True)
    train_labels = label_arr[train_mask.to_numpy()]
    valid_labels = label_arr[valid_mask.to_numpy()]

    if CFG.max_train_rows is not None:
        keep = min(CFG.max_train_rows, len(train_frame))
        train_frame = train_frame.sample(keep, random_state=CFG.random_seed).reset_index(drop=True)
        train_labels = build_multihot(train_frame['labels'], label2idx)
    if CFG.max_valid_rows is not None:
        keep = min(CFG.max_valid_rows, len(valid_frame))
        valid_frame = valid_frame.sample(keep, random_state=CFG.random_seed).reset_index(drop=True)
        valid_labels = build_multihot(valid_frame['labels'], label2idx)

    return train_frame, valid_frame, train_labels.astype(np.float32), valid_labels.astype(np.float32)


train_frame_labeled, valid_frame, train_labels_fold, valid_labels_fold = build_fold_frame(labeled_df, labeled_arr, CFG.fold)
valid_labeled_soundscape_files = set(valid_frame.loc[valid_frame['source'] == 'soundscape_clip', 'filename'].tolist())
all_labeled_soundscape_files = set(soundscape_clip_manifest['filename'].tolist())
teacher_folds = [fold for fold in range(CFG.n_folds) if fold != CFG.fold]

print({
    'fold': CFG.fold,
    'teacher_folds': teacher_folds,
    'train_rows_labeled': int(len(train_frame_labeled)),
    'valid_rows': int(len(valid_frame)),
    'valid_soundscape_rows': int((valid_frame['source'] == 'soundscape_clip').sum()),
})


{'fold': 3, 'teacher_folds': [0, 1, 2], 'train_rows_labeled': 27127, 'valid_rows': 8951, 'valid_soundscape_rows': 136}


In [16]:
def parse_soundscape_filename(name: str) -> dict[str, tp.Any]:
    stem = Path(name).stem
    parts = stem.split('_')
    site = parts[3] if len(parts) >= 4 else None
    time_token = parts[-1] if parts else '000000'
    hour_utc = int(time_token[:2]) if time_token.isdigit() and len(time_token) >= 2 else -1
    return {'site': site, 'hour_utc': hour_utc}


def build_unlabeled_soundscape_manifest() -> pd.DataFrame:
    rows = []
    file_paths = sorted(TRAIN_SOUNDSCAPE_DIR.glob('*.ogg'))
    unlabeled_paths = [p for p in file_paths if p.name not in all_labeled_soundscape_files and p.name not in valid_labeled_soundscape_files]
    if CFG.max_pseudo_files is not None:
        unlabeled_paths = unlabeled_paths[: CFG.max_pseudo_files]
    for file_path in tqdm(unlabeled_paths, desc='build_pseudo_manifest'):
        info = sf.info(file_path)
        duration_sec = int(math.floor(info.frames / info.samplerate)) if info.samplerate else 0
        if duration_sec < CFG.segment_seconds:
            continue
        max_end = duration_sec - (duration_sec % CFG.segment_seconds)
        meta = parse_soundscape_filename(file_path.name)
        for end_sec in range(CFG.segment_seconds, max_end + 1, CFG.segment_seconds):
            start_sec = end_sec - CFG.segment_seconds
            row_id = f"{Path(file_path.name).stem}_{end_sec}"
            rows.append({
                'filename': file_path.name,
                'row_id': row_id,
                'file_path': str(file_path),
                'start_sec': int(start_sec),
                'end_sec': int(end_sec),
                'site': meta['site'],
                'hour_utc': int(meta['hour_utc']),
            })
    return pd.DataFrame(rows)


def read_full_audio(file_path: Path) -> np.ndarray:
    with sf.SoundFile(file_path) as snd:
        audio = snd.read(dtype='float32', always_2d=True)
    audio = audio.mean(axis=1).astype(np.float32, copy=False)
    return np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)


def slice_segment_from_audio(audio: np.ndarray, start_sec: int) -> np.ndarray:
    start = int(start_sec * CFG.sample_rate)
    end = start + CFG.clip_samples
    clip = audio[start:end]
    if len(clip) < CFG.clip_samples:
        clip = np.pad(clip, (0, CFG.clip_samples - len(clip)))
    return clip.astype(np.float32)

pseudo_manifest_path = OUTPUT_DIR / 'pseudo_manifest.parquet'
if RUN_PREPARE:
    pseudo_manifest = build_unlabeled_soundscape_manifest()
    pseudo_manifest.to_parquet(pseudo_manifest_path, index=False)
else:
    pseudo_manifest = pd.read_parquet(pseudo_manifest_path)

print({
    'pseudo_manifest_rows': int(len(pseudo_manifest)),
    'pseudo_manifest_files': int(pseudo_manifest['filename'].nunique()) if len(pseudo_manifest) else 0,
    'teacher_folds': teacher_folds,
})


{'pseudo_manifest_rows': 127104, 'pseudo_manifest_files': 10592, 'teacher_folds': [0, 1, 2]}


In [17]:
class LogMelSpectrogramTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate,
            n_fft=CFG.n_fft,
            win_length=CFG.win_length,
            hop_length=CFG.hop_length,
            f_min=CFG.f_min,
            n_mels=CFG.n_mels,
            power=2.0,
            center=True,
            pad_mode='reflect',
            norm='slaney',
            mel_scale='htk',
        )
        self.db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=CFG.top_db)

    @torch.no_grad()
    def forward(self, wave: torch.Tensor) -> torch.Tensor:
        if wave.ndim == 1:
            wave = wave.unsqueeze(0)
        mel = self.mel(wave)
        mel = self.db(mel)
        mel = mel.unsqueeze(1)
        mel = F.interpolate(mel, size=CFG.image_size, mode='bilinear', align_corners=False)
        flat = mel.flatten(start_dim=1)
        min_val = flat.min(dim=1).values[:, None, None, None]
        max_val = flat.max(dim=1).values[:, None, None, None]
        mel = (mel - min_val) / (max_val - min_val + 1e-7)
        return mel


class SpectrogramMixUp(nn.Module):
    def __init__(self, alpha: float = 1.0, theta: float = 0.8):
        super().__init__()
        self.beta_dist = torch.distributions.beta.Beta(alpha, alpha)
        self.theta = theta

    def forward(self, image: torch.Tensor, label: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        batch_size = image.shape[0]
        if batch_size < 2:
            return image, label
        lambdas = self.beta_dist.sample((batch_size,)).to(image.device)
        lambdas = torch.maximum(lambdas, 1 - lambdas).float()
        perm = torch.randperm(batch_size, device=image.device)
        image = lambdas[:, None, None, None] * image + (1 - lambdas[:, None, None, None]) * image[perm]
        label = lambdas[:, None] * label + (1 - lambdas[:, None]) * label[perm]
        label = torch.where(label >= self.theta, torch.ones_like(label), label)
        return image, label


def build_model() -> nn.Module:
    return timm.create_model(
        CFG.model_name,
        pretrained=CFG.pretrained,
        in_chans=1,
        num_classes=N_CLASSES,
        drop_path_rate=CFG.drop_path_rate,
    )


def load_pretrained_state(model: nn.Module, checkpoint_path: Path) -> None:
    payload = torch.load(checkpoint_path, map_location='cpu')
    state_dict = payload['model_state_dict'] if 'model_state_dict' in payload else payload
    model.load_state_dict(state_dict, strict=True)


def read_audio_region(path: str, clip_start_frame: int, clip_end_frame: int, sample_frames: int, training: bool) -> np.ndarray:
    with sf.SoundFile(path) as snd:
        total_frames = snd.frames
        region_start = max(int(clip_start_frame), 0)
        region_end = total_frames if int(clip_end_frame) <= 0 else min(int(clip_end_frame), total_frames)
        region_frames = max(region_end - region_start, 0)

        if region_frames <= 0:
            return np.zeros(sample_frames, dtype=np.float32)

        if region_frames >= sample_frames:
            offset = np.random.randint(region_frames - sample_frames + 1) if training else 0
            snd.seek(region_start + offset)
            wave = snd.read(frames=sample_frames, dtype='float32', always_2d=True)
            wave = wave.mean(axis=1).astype(np.float32, copy=False)
            if wave.shape[0] == sample_frames:
                return wave
        else:
            snd.seek(region_start)
            wave = snd.read(frames=region_frames, dtype='float32', always_2d=True)
            wave = wave.mean(axis=1).astype(np.float32, copy=False)

        actual_frames = int(wave.shape[0])
        if actual_frames >= sample_frames:
            return wave[:sample_frames]

        padded = np.zeros(sample_frames, dtype=np.float32)
        pad_start = np.random.randint(sample_frames - actual_frames + 1) if training else 0
        padded[pad_start: pad_start + actual_frames] = wave
        return padded


@torch.no_grad()
def build_teacher_models() -> tuple[list[nn.Module], nn.Module]:
    models = []
    mel_transform = LogMelSpectrogramTransform().to(device).eval()
    for fold in teacher_folds:
        ckpt = EXP011_DIR / f'fold_{fold:02d}' / 'best_model.pt'
        model = build_model().to(device)
        load_pretrained_state(model, ckpt)
        model.eval()
        models.append(model)
    return models, mel_transform


In [18]:
def generate_pseudo_labels(manifest_df: pd.DataFrame, output_dir: Path) -> tuple[pd.DataFrame, np.ndarray]:
    teacher_models, mel_transform = build_teacher_models()
    rows = []
    probs_all = []

    for filename, file_df in tqdm(manifest_df.groupby('filename', sort=False), total=manifest_df['filename'].nunique(), desc='pseudo_label_files'):
        file_df = file_df.sort_values('end_sec').reset_index(drop=True)
        audio = read_full_audio(Path(file_df.iloc[0]['file_path']))
        segments = np.stack([slice_segment_from_audio(audio, int(start_sec)) for start_sec in file_df['start_sec']], axis=0)

        ensemble_probs = []
        for teacher in teacher_models:
            probs_parts = []
            for start in range(0, len(segments), CFG.pseudo_batch_size):
                batch_wave = torch.from_numpy(segments[start: start + CFG.pseudo_batch_size]).to(device, dtype=torch.float32)
                image = mel_transform(batch_wave)
                with autocast_context():
                    logits = teacher(image)
                probs_parts.append(torch.sigmoid(logits).detach().float().cpu().numpy().astype(np.float32))
            ensemble_probs.append(np.concatenate(probs_parts, axis=0))

        probs = np.mean(np.stack(ensemble_probs, axis=0), axis=0)
        probs = np.power(np.clip(probs, 1e-6, 1.0), CFG.pseudo_power).astype(np.float32)
        confidence = probs.max(axis=1)
        keep_mask = confidence >= CFG.pseudo_min_confidence

        kept_df = file_df.loc[keep_mask].copy().reset_index(drop=True)
        kept_probs = probs[keep_mask]
        kept_conf = confidence[keep_mask]

        if CFG.max_pseudo_segments_per_file is not None and len(kept_df) > CFG.max_pseudo_segments_per_file:
            order = np.argsort(-kept_conf)[: CFG.max_pseudo_segments_per_file]
            kept_df = kept_df.iloc[order].reset_index(drop=True)
            kept_probs = kept_probs[order]
            kept_conf = kept_conf[order]

        for row_idx in range(len(kept_df)):
            row = kept_df.iloc[row_idx].to_dict()
            row['pseudo_confidence'] = float(kept_conf[row_idx])
            row['teacher_folds'] = ','.join(str(fold) for fold in teacher_folds)
            row['pseudo_index'] = len(probs_all)
            rows.append(row)
            probs_all.append(kept_probs[row_idx])

    pseudo_meta = pd.DataFrame(rows)
    if CFG.max_pseudo_segments_total is not None and len(pseudo_meta) > CFG.max_pseudo_segments_total:
        pseudo_meta = pseudo_meta.sort_values('pseudo_confidence', ascending=False).head(CFG.max_pseudo_segments_total).reset_index(drop=True)
        keep_indices = pseudo_meta['pseudo_index'].tolist()
        pseudo_probs_arr = np.stack([probs_all[idx] for idx in keep_indices], axis=0)
        pseudo_meta['pseudo_index'] = np.arange(len(pseudo_meta), dtype=np.int32)
    else:
        pseudo_probs_arr = np.stack(probs_all, axis=0) if probs_all else np.zeros((0, N_CLASSES), dtype=np.float32)

    pseudo_meta.to_parquet(output_dir / 'pseudo_label_meta.parquet', index=False)
    np.savez_compressed(output_dir / 'pseudo_label_probs.npz', probs=pseudo_probs_arr.astype(np.float32))

    summary = {
        'teacher_folds': teacher_folds,
        'pseudo_rows': int(len(pseudo_meta)),
        'pseudo_files': int(pseudo_meta['filename'].nunique()) if len(pseudo_meta) else 0,
        'confidence_mean': None if len(pseudo_meta) == 0 else float(pseudo_meta['pseudo_confidence'].mean()),
        'confidence_p75': None if len(pseudo_meta) == 0 else float(pseudo_meta['pseudo_confidence'].quantile(0.75)),
        'pseudo_min_confidence': float(CFG.pseudo_min_confidence),
        'max_pseudo_segments_per_file': int(CFG.max_pseudo_segments_per_file),
    }
    save_json(summary, output_dir / 'teacher_summary.json')
    return pseudo_meta, pseudo_probs_arr.astype(np.float32)


pseudo_meta_path = OUTPUT_DIR / 'pseudo_label_meta.parquet'
pseudo_probs_path = OUTPUT_DIR / 'pseudo_label_probs.npz'
if pseudo_meta_path.exists() and pseudo_probs_path.exists():
    pseudo_meta_df = pd.read_parquet(pseudo_meta_path)
    pseudo_probs = np.load(pseudo_probs_path)['probs'].astype(np.float32)
else:
    pseudo_meta_df = pd.DataFrame(columns=['row_id'])
    pseudo_probs = np.zeros((0, N_CLASSES), dtype=np.float32)

if RUN_PSEUDO_GENERATION:
    pseudo_meta_df, pseudo_probs = generate_pseudo_labels(pseudo_manifest, OUTPUT_DIR)
    print(json.dumps(json.loads((OUTPUT_DIR / 'teacher_summary.json').read_text()), indent=2))
else:
    print({
        'existing_pseudo_rows': int(len(pseudo_meta_df)),
        'existing_pseudo_files': int(pseudo_meta_df['filename'].nunique()) if len(pseudo_meta_df) else 0,
    })


{'existing_pseudo_rows': 20595, 'existing_pseudo_files': 5293}


In [19]:
class LabeledDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, label_arr: np.ndarray, training: bool):
        self.frame = frame.reset_index(drop=True)
        self.labels = label_arr.astype(np.float32)
        self.training = training
        self.sample_frames = CFG.clip_samples

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict[str, tp.Any]:
        row = self.frame.iloc[idx]
        wave = read_audio_region(
            path=str(row['file_path']),
            clip_start_frame=int(row['clip_start_frame']),
            clip_end_frame=int(row['clip_end_frame']),
            sample_frames=self.sample_frames,
            training=self.training,
        )
        return {
            'wave': wave,
            'target': self.labels[idx],
            'weight': 1.0,
            'index': idx,
            'source': row['source'],
        }


class PseudoLabelDataset(Dataset):
    def __init__(self, meta_df: pd.DataFrame, probs: np.ndarray):
        self.meta_df = meta_df.reset_index(drop=True)
        self.probs = probs.astype(np.float32)
        self.sample_frames = CFG.clip_samples

    def __len__(self) -> int:
        return len(self.meta_df)

    def __getitem__(self, idx: int) -> dict[str, tp.Any]:
        row = self.meta_df.iloc[idx]
        audio = read_full_audio(Path(row['file_path']))
        wave = slice_segment_from_audio(audio, int(row['start_sec']))
        weight = float(CFG.pseudo_loss_weight * max(float(row['pseudo_confidence']) ** CFG.pseudo_sampler_power, 1e-3))
        return {
            'wave': wave,
            'target': self.probs[int(row['pseudo_index'])],
            'weight': weight,
            'index': idx,
            'source': 'pseudo',
        }


def make_loader(dataset: Dataset, training: bool, sampler=None) -> DataLoader:
    kwargs = dict(
        dataset=dataset,
        batch_size=CFG.batch_size,
        shuffle=(training and sampler is None),
        sampler=sampler,
        drop_last=False,
        num_workers=CFG.num_workers,
        pin_memory=(device.type == 'cuda'),
    )
    return DataLoader(**kwargs)


def compute_macro_auc(y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float | int]:
    positive_mask = y_true.sum(axis=0) > 0
    negative_mask = (y_true.shape[0] - y_true.sum(axis=0)) > 0
    scored_mask = positive_mask & negative_mask
    scored_classes = int(scored_mask.sum())
    skipped_no_positive = int((~positive_mask).sum())
    skipped_no_negative = int((~negative_mask).sum())
    if scored_classes == 0:
        return {
            'macro_auc': np.nan,
            'scored_classes': scored_classes,
            'skipped_no_positive': skipped_no_positive,
            'skipped_no_negative': skipped_no_negative,
        }
    macro_auc = roc_auc_score(y_true[:, scored_mask], y_score[:, scored_mask], average='macro')
    return {
        'macro_auc': float(macro_auc),
        'scored_classes': scored_classes,
        'skipped_no_positive': skipped_no_positive,
        'skipped_no_negative': skipped_no_negative,
    }


def evaluate_predictions(valid_frame: pd.DataFrame, y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float | int]:
    overall = compute_macro_auc(y_true, y_score)
    soundscape_mask = valid_frame['source'].eq('soundscape_clip').to_numpy()
    soundscape_metrics = {
        'soundscape_macro_auc': np.nan,
        'soundscape_scored_classes': 0,
        'soundscape_skipped_no_positive': 0,
        'soundscape_skipped_no_negative': 0,
    }
    if soundscape_mask.any():
        tmp = compute_macro_auc(y_true[soundscape_mask], y_score[soundscape_mask])
        soundscape_metrics = {
            'soundscape_macro_auc': tmp['macro_auc'],
            'soundscape_scored_classes': tmp['scored_classes'],
            'soundscape_skipped_no_positive': tmp['skipped_no_positive'],
            'soundscape_skipped_no_negative': tmp['skipped_no_negative'],
        }
    return {**overall, **soundscape_metrics}


def save_checkpoint(path: Path, model: nn.Module, optimizer, scheduler, scaler, epoch: int, metrics: dict[str, tp.Any], history: list[dict[str, tp.Any]], resume_mode: str):
    payload = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict() if amp_enabled else None,
        'metrics': metrics,
        'cfg': asdict(CFG),
        'history': history,
        'resume_mode': resume_mode,
    }
    torch.save(payload, path)


In [20]:
RUN_SUMMARY = {
    'fold': CFG.fold,
    'teacher_folds': teacher_folds,
    'pseudo_rows_available': int(len(pseudo_meta_df)),
    'pseudo_files_available': int(pseudo_meta_df['filename'].nunique()) if len(pseudo_meta_df) else 0,
}
print(json.dumps(RUN_SUMMARY, indent=2))

if RUN_TRAINING:
    train_dataset_labeled = LabeledDataset(train_frame_labeled, train_labels_fold, training=True)
    valid_dataset = LabeledDataset(valid_frame, valid_labels_fold, training=False)

    train_parts = [train_dataset_labeled]
    sample_weights = [1.0] * len(train_dataset_labeled)
    if len(pseudo_meta_df):
        pseudo_dataset = PseudoLabelDataset(pseudo_meta_df, pseudo_probs)
        train_parts.append(pseudo_dataset)
        pseudo_weights = np.clip(pseudo_meta_df['pseudo_confidence'].to_numpy(dtype=np.float32), 1e-3, 1.0) ** CFG.pseudo_sampler_power
        pseudo_weights = (CFG.pseudo_loss_weight * pseudo_weights).tolist()
        sample_weights.extend(pseudo_weights)
    else:
        pseudo_dataset = None

    train_dataset = train_parts[0] if len(train_parts) == 1 else ConcatDataset(train_parts)
    sampler = WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)
    train_loader = make_loader(train_dataset, training=True, sampler=sampler)
    valid_loader = make_loader(valid_dataset, training=False)

    mel_transform = LogMelSpectrogramTransform().to(device).eval()
    mixup = SpectrogramMixUp(alpha=CFG.mixup_alpha, theta=CFG.mixup_theta)
    model = build_model().to(device)
    init_ckpt = EXP011_DIR / f'fold_{CFG.fold:02d}' / 'best_model.pt'
    load_pretrained_state(model, init_ckpt)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=max(1, CFG.epochs))
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

    history_path = OUTPUT_DIR / 'history.csv'
    best_ckpt_path = OUTPUT_DIR / 'best_model.pt'
    last_ckpt_path = OUTPUT_DIR / 'last_model.pt'

    history: list[dict[str, tp.Any]] = []
    start_epoch = 1
    best_metric = -float('inf')
    best_epoch = None
    resume_mode = 'init_from_exp011_fold'

    if RESUME_TRAINING and last_ckpt_path.exists():
        payload = torch.load(last_ckpt_path, map_location='cpu')
        model.load_state_dict(payload['model_state_dict'])
        optimizer.load_state_dict(payload['optimizer_state_dict'])
        scheduler.load_state_dict(payload['scheduler_state_dict'])
        if amp_enabled and payload.get('scaler_state_dict') is not None:
            scaler.load_state_dict(payload['scaler_state_dict'])
        history = payload.get('history', [])
        start_epoch = int(payload.get('epoch', 0)) + 1
        best_metric = max([float(row.get('selection_metric', -float('inf'))) for row in history], default=-float('inf'))
        for row in history:
            if float(row.get('selection_metric', -float('inf'))) == best_metric:
                best_epoch = int(row['epoch'])
        resume_mode = 'resume_exp014'

    for epoch in range(start_epoch, CFG.epochs + 1):
        model.train()
        train_loss_sum = 0.0
        for batch in tqdm(train_loader, desc=f'epoch {epoch} train', leave=False):
            wave = batch['wave'].to(device, dtype=torch.float32)
            target = batch['target'].to(device, dtype=torch.float32)
            weight = batch['weight'].to(device, dtype=torch.float32)
            optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                image = mel_transform(wave)
            if CFG.use_mixup:
                image, target = mixup(image, target)
            with autocast_context():
                logits = model(image)
                raw_loss = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
                sample_loss = raw_loss.mean(dim=1) * weight
                loss = sample_loss.sum() / weight.sum().clamp_min(1e-6)
            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            train_loss_sum += float(loss.item())
        scheduler.step()
        train_loss = train_loss_sum / max(1, len(train_loader))

        model.eval()
        valid_loss_sum = 0.0
        logits_parts = []
        label_parts = []
        index_parts = []
        for batch in tqdm(valid_loader, desc=f'epoch {epoch} valid', leave=False):
            wave = batch['wave'].to(device, dtype=torch.float32)
            target = batch['target'].to(device, dtype=torch.float32)
            with torch.no_grad():
                image = mel_transform(wave)
                with autocast_context():
                    logits = model(image)
                    loss = F.binary_cross_entropy_with_logits(logits, target)
            valid_loss_sum += float(loss.item())
            logits_parts.append(logits.detach().float().cpu().numpy())
            label_parts.append(target.detach().float().cpu().numpy())
            index_parts.append(batch['index'].detach().cpu().numpy())
        valid_loss = valid_loss_sum / max(1, len(valid_loader))

        logits_all = np.concatenate(logits_parts, axis=0)
        labels_all = np.concatenate(label_parts, axis=0)
        index_all = np.concatenate(index_parts, axis=0)
        probs_all = 1.0 / (1.0 + np.exp(-logits_all))
        order = np.argsort(index_all)
        labels_all = labels_all[order]
        probs_all = probs_all[order]

        valid_metrics = evaluate_predictions(valid_frame.reset_index(drop=True), labels_all, probs_all)
        selection_metric = valid_metrics['soundscape_macro_auc']
        if np.isnan(selection_metric):
            selection_metric = valid_metrics['macro_auc']

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'macro_auc': valid_metrics['macro_auc'],
            'scored_classes': valid_metrics['scored_classes'],
            'skipped_no_positive': valid_metrics['skipped_no_positive'],
            'skipped_no_negative': valid_metrics['skipped_no_negative'],
            'soundscape_macro_auc': valid_metrics['soundscape_macro_auc'],
            'soundscape_scored_classes': valid_metrics['soundscape_scored_classes'],
            'soundscape_skipped_no_positive': valid_metrics['soundscape_skipped_no_positive'],
            'soundscape_skipped_no_negative': valid_metrics['soundscape_skipped_no_negative'],
            'valid_loss': valid_loss,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'selection_metric': float(selection_metric),
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(row)

        save_checkpoint(last_ckpt_path, model, optimizer, scheduler, scaler, epoch, row, history, resume_mode)
        if selection_metric > best_metric:
            best_metric = float(selection_metric)
            best_epoch = epoch
            save_checkpoint(best_ckpt_path, model, optimizer, scheduler, scaler, epoch, row, history, resume_mode)
        if CFG.save_every_epoch:
            save_checkpoint(OUTPUT_DIR / f'epoch_{epoch:02d}.pt', model, optimizer, scheduler, scaler, epoch, row, history, resume_mode)

    snapshot = {
        'experiment_id': CFG.experiment_id,
        'experiment_name': CFG.experiment_name,
        'fold': CFG.fold,
        'best_epoch': best_epoch,
        'best_selection_metric': best_metric,
        'best_macro_auc': max((float(row['macro_auc']) for row in history), default=float('nan')),
        'best_soundscape_macro_auc': max((float(row['soundscape_macro_auc']) for row in history if pd.notna(row['soundscape_macro_auc'])), default=float('nan')),
        'train_rows_labeled': int(len(train_frame_labeled)),
        'valid_rows': int(len(valid_frame)),
        'pseudo_rows': int(len(pseudo_meta_df)),
        'pseudo_files': int(pseudo_meta_df['filename'].nunique()) if len(pseudo_meta_df) else 0,
        'teacher_folds': teacher_folds,
        'output_dir': str(OUTPUT_DIR),
        'resume_mode': resume_mode,
    }
    save_json(snapshot, OUTPUT_DIR / 'result_snapshot.json')
    print(json.dumps(snapshot, indent=2))
else:
    print('Training skipped. Set RUN_TRAINING = True after pseudo cache is ready.')


{
  "fold": 3,
  "teacher_folds": [
    0,
    1,
    2
  ],
  "pseudo_rows_available": 20595,
  "pseudo_files_available": 5293
}


epoch 1 train:   0%|          | 0/2983 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [16, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]
/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [10, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero el

epoch 1 valid:   0%|          | 0/560 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [7, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]


...

epoch 8 train:   0%|          | 0/2983 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [16, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]
/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [10, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero el

epoch 8 valid:   0%|          | 0/560 [00:00<?, ?it/s]

/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/.venv/lib/python3.14/site-packages/torch/functional.py:681: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [7, 512, 1025]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  return _VF.stft(  # type: ignore[attr-defined]
/var/folders/kq/gkktf1p90g52bfdjq8s8qcdm0000gp/T/ipykernel_77665/1848022932.py:113: RuntimeWarning: overflow encountered in exp
  probs_all = 1.0 / (1.0 + np.exp(-logits_all))


{'epoch': 8, 'train_loss': 0.004694443747030957, 'macro_auc': 0.966097202442945, 'scored_classes': 208, 'skipped_no_positive': 26, 'skipped_no_negative': 0, 'soundscape_macro_auc': 0.760823009884516, 'soundscape_scored_classes': 40, 'soundscape_skipped_no_positive': 194, 'soundscape_skipped_no_negative': 0, 'valid_loss': 0.013497053021715796, 'learning_rate': 0.0, 'selection_metric': 0.760823009884516}
{
  "experiment_id": "exp_014",
  "experiment_name": "hgnetv2_pseudolabel",
  "fold": 3,
  "best_epoch": 3,
  "best_selection_metric": 0.7702235930754797,
  "best_macro_auc": 0.9661132491858939,
  "best_soundscape_macro_auc": 0.7702235930754797,
  "train_rows_labeled": 27127,
  "valid_rows": 8951,
  "pseudo_rows": 20595,
  "pseudo_files": 5293,
  "teacher_folds": [
    0,
    1,
    2
  ],
  "output_dir": "/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_014_hgnetv2_pseudolabel/fold_03",
  "resume_mode": "init_from_exp011_fold"
}
